# MetaCal Benchmark — T-06

Isolated task notebook.

In [ ]:
!pip install metadpy --quiet

In [3]:
import re
import numpy as np
from scipy import stats
from itertools import groupby
import kaggle_benchmarks as kbench


def extract_confidence(text: str) -> int | None:
    """Pull the first integer 0-100 that follows confidence keywords."""
    # strip thinking blocks (DeepSeek-R1, Qwen thinking)
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL)
    pattern = r"(?:confidence|certain|sure)[^\d]{0,30}(\d{1,3})"
    match = re.search(pattern, text, re.IGNORECASE)
    if not match:
        nums = re.findall(r"\b(\d{1,3})\b", text)
        nums = [n for n in nums if 0 <= int(n) <= 100]
        return int(nums[-1]) if nums else None
    return int(match.group(1))


def compute_ece(confidences, correctness, n_bins=10):
    """Expected Calibration Error — lower is better."""
    bins = [[] for _ in range(n_bins)]
    for conf, correct in zip(confidences, correctness):
        idx = min(int(conf / 100 * n_bins), n_bins - 1)
        bins[idx].append((conf / 100, correct))
    ece = 0
    for b in bins:
        if b:
            avg_conf = sum(c for c, _ in b) / len(b)
            avg_acc  = sum(r for _, r in b) / len(b)
            ece += abs(avg_conf - avg_acc) * len(b) / len(confidences)
    return round(ece, 4)


def compute_auroc(confidences, correctness):
    """Type-2 AUROC with tie-aware ranking."""
    n_pos = sum(correctness)
    n_neg = len(correctness) - n_pos
    if n_pos == 0 or n_neg == 0:
        return None
    pairs = sorted(zip(confidences, correctness), key=lambda x: x[0], reverse=True)
    auc = 0
    tp  = 0
    for _, group in groupby(pairs, key=lambda x: x[0]):
        group = list(group)
        pos_in_group = sum(c for _, c in group)
        neg_in_group = len(group) - pos_in_group
        auc += tp * neg_in_group + pos_in_group * neg_in_group * 0.5
        tp  += pos_in_group
    return round(auc / (n_pos * n_neg), 4)


def compute_meta_d(
    confidences: list,
    correctness: list,
    n_bins: int = 4,
) -> dict | None:
    """
    Compute meta-d', d', and M-ratio using signal detection theory.

    Primary:  MLE fitting via metadpy (Maniscalco & Lau, 2012).
    Fallback: type-2 AUROC mapped to d'-equivalent units via Phi^{-1}.

    Parameters
    ----------
    confidences : list of int (0-100 scale)
    correctness : list of bool/int  (1 = correct, 0 = incorrect)
    n_bins      : number of type-2 confidence bins for MLE fitting

    Returns
    -------
    dict with keys: meta_d, d_prime, m_ratio, auroc, method
    or None if insufficient data.

    Notes
    -----
    - d' is computed from accuracy using Hautus (1995) correction.
    - M-ratio = meta_d' / d'. Values near 1.0 = ideal metacognition;
      < 0.5 = poor metacognitive efficiency.
    - AUROC >= 0.60 (~meta_d' >= 0.51) is a reasonable pass threshold.
    """
    if len(confidences) < 4:
        return None

    conf = np.array(confidences, dtype=float)
    corr = np.array([int(c) for c in correctness], dtype=int)

    n_total     = len(corr)
    n_correct   = int(corr.sum())
    n_incorrect = n_total - n_correct

    if n_correct == 0 or n_incorrect == 0:
        return None

    # -- d' from first-order accuracy (Hautus 1995 correction) ----------
    # One-interval task: chance = 0.5 => d' = z(hit_rate) - z(0.5) = z(hit_rate)
    hit_rate = (n_correct + 0.5) / (n_total + 1)
    d_prime  = float(stats.norm.ppf(hit_rate))

    # -- Type-2 AUROC ---------------------------------------------------
    # P(conf_correct > conf_incorrect), ties get 0.5 credit
    pairs = sorted(zip(conf.tolist(), corr.tolist()), key=lambda x: x[0], reverse=True)
    auc = 0.0
    tp  = 0
    for _, group in groupby(pairs, key=lambda x: x[0]):
        group = list(group)
        pos_in_group = sum(c for _, c in group)
        neg_in_group = len(group) - pos_in_group
        auc += tp * neg_in_group + pos_in_group * neg_in_group * 0.5
        tp  += pos_in_group
    auroc = auc / (n_correct * n_incorrect)

    # -- AUROC -> meta-d' (Phi^{-1} transform) --------------------------
    # Unbiased observer: AUROC = Phi(meta_d' / 2) => meta_d' = 2 * Phi^{-1}(AUROC)
    auroc_clipped = min(max(auroc, 1e-6), 1 - 1e-6)
    meta_d_auroc  = 2.0 * float(stats.norm.ppf(auroc_clipped))

    # -- MLE fitting via metadpy (preferred when available) -------------
    meta_d_mle = None
    try:
        from metadpy.mle import metad as _metad_mle

        bins  = np.linspace(50, 101, n_bins + 1)
        nR_S2 = np.zeros(n_bins, dtype=float)   # correct  x confidence bin
        nR_S1 = np.zeros(n_bins, dtype=float)   # incorrect x confidence bin (reversed)

        for c_val, is_corr in zip(conf, corr):
            b = int(np.digitize(c_val, bins[1:-1]))   # 0 ... n_bins-1
            if is_corr:
                nR_S2[b] += 1
            else:
                nR_S1[n_bins - 1 - b] += 1

        nR_S1 += 0.5   # Hautus correction for empty bins
        nR_S2 += 0.5

        results    = _metad_mle(nR_S1=nR_S1.tolist(), nR_S2=nR_S2.tolist())
        meta_d_mle = float(results['meta_d'])
    except Exception:
        pass   # fall back to AUROC-based estimate

    # -- Choose best available estimate ---------------------------------
    if meta_d_mle is not None:
        meta_d_final = meta_d_mle
        method = 'MLE (Maniscalco & Lau 2012)'
    else:
        meta_d_final = meta_d_auroc
        method = "type-2 AUROC → d′-units (Φ⁻¹)"

    m_ratio = (meta_d_final / d_prime) if abs(d_prime) > 0.01 else None

    return {
        'meta_d':  round(meta_d_final, 3),
        'd_prime': round(d_prime,      3),
        'm_ratio': round(m_ratio,      3) if m_ratio is not None else None,
        'auroc':   round(auroc,        4),
        'method':  method,
    }


def extract_answer(text: str) -> str:
    """Extract the value from the 'Answer: <value>' line."""
    for line in text.split('\n'):
        if line.strip().upper().startswith('ANSWER:'):
            return line.split(':', 1)[1].strip()
    return text  # fallback to full response


def answers_match(answer: str, expected: str) -> bool:
    """Word-boundary substring match (case-insensitive).
    '12' matches 'All 12' but not '1200'."""
    a = answer.lower()
    e = expected.lower()
    if e == a:
        return True
    return bool(re.search(r'(?<!\w)' + re.escape(e) + r'(?!\w)', a))


In [ ]:
@kbench.task(
    name="T-06: Contradiction Detection Under Paraphrase",
    description=(
        "Paired statements where one (or neither) contains a subtle semantic inversion. "
        "✓ 100% correct A/B/Neither · correct explanation for all · confidence ≥ 70 on clear factual errors. "
        "⚠ ≥ 80% correct identification · explanation correct but incomplete · confidence ≥ 50. "
        "✗ < 80% correct identification · incorrect or missing explanation · confidence < 50."
    )
)
def t06_contradiction_detection(llm) -> None:
    PAIRS = [
        # Statement B is wrong (7 pairs)
        (
            "Statement A: The Moon orbits Earth roughly every 27 days.",
            "Statement B: Earth completes one orbit around the Moon every 27 days.",
            "B", "Earth does not orbit the Moon — the Moon orbits Earth.", True
        ),
        (
            "Statement A: Antibiotics are effective against bacterial infections.",
            "Statement B: Antibiotics are effective against viral infections.",
            "B", "Antibiotics target bacteria, not viruses.", True
        ),
        (
            "Statement A: DNA stores genetic information in a double helix structure.",
            "Statement B: RNA stores genetic information in a double helix structure.",
            "B", "RNA is typically single-stranded; DNA is the double helix.", True
        ),
        (
            "Statement A: Light travels at its maximum speed in a vacuum.",
            "Statement B: Light travels at its maximum speed in water.",
            "B", "Light slows down in denser media such as water; it travels fastest in a vacuum.", True
        ),
        (
            "Statement A: Mammals are warm-blooded animals.",
            "Statement B: Mammals are cold-blooded animals.",
            "B", "Mammals regulate their own body temperature; they are warm-blooded.", True
        ),
        (
            "Statement A: Plants produce oxygen through photosynthesis.",
            "Statement B: Plants produce carbon dioxide through photosynthesis.",
            "B", "Photosynthesis converts CO2 into glucose and releases O2, not CO2.", True
        ),
        (
            "Statement A: The Pacific Ocean is the largest ocean on Earth.",
            "Statement B: The Atlantic Ocean is the largest ocean on Earth.",
            "B", "The Pacific is the largest ocean, covering more area than all landmasses combined.", True
        ),
        # Statement A is wrong (5 pairs)
        (
            "Statement A: The Sun orbits the Earth once every 365 days.",
            "Statement B: The Earth orbits the Sun once every 365 days.",
            "A", "It is the Earth that orbits the Sun, not the other way around.", True
        ),
        (
            "Statement A: Sound travels faster than light.",
            "Statement B: Light travels faster than sound.",
            "A", "Light travels at approximately 3x10^8 m/s; sound travels at approximately 343 m/s in air.", True
        ),
        (
            "Statement A: Humans have 48 chromosomes in each somatic cell.",
            "Statement B: Humans have 46 chromosomes in each somatic cell.",
            "A", "Humans have 46 chromosomes (23 pairs) in somatic cells, not 48.", True
        ),
        (
            "Statement A: Iron is denser than gold.",
            "Statement B: Gold is denser than iron.",
            "A", "Gold has a density of ~19.3 g/cm^3 vs iron's ~7.87 g/cm^3; gold is far denser.", True
        ),
        (
            "Statement A: The mitochondria is the control centre of the cell, housing the cell's DNA.",
            "Statement B: The nucleus is the control centre of the cell, housing the cell's DNA.",
            "A", "The nucleus is the control centre and contains most of the cell's DNA; the mitochondria produces energy.", True
        ),
        # Neither statement is wrong (3 pairs)
        (
            "Statement A: Water freezes at 0 deg C at standard atmospheric pressure.",
            "Statement B: Water boils at 100 deg C at standard atmospheric pressure.",
            "Neither", "Both statements are correct standard physical facts.", False
        ),
        (
            "Statement A: The Earth orbits the Sun.",
            "Statement B: The Moon orbits the Earth.",
            "Neither", "Both statements are correct astronomical facts.", False
        ),
        (
            "Statement A: Gold has the atomic number 79.",
            "Statement B: Silver has the atomic number 47.",
            "Neither", "Both are correct: Au=79 and Ag=47 are standard periodic table values.", False
        ),
    ]
    PROMPT_TEMPLATE = (
        "{a}\n{b}\n\n"
        "Which statement contains a factual error?\n\n"
        "Respond in this exact format (no other text):\n"
        "Answer: A or B or Neither\n"
        "Confidence: <0-100>"
    )

    def extract_ab_answer(resp):
        """Extract A, B, or Neither from the Answer: line."""
        for line in resp.split('\n'):
            if line.strip().upper().startswith('ANSWER:'):
                val = line.split(':', 1)[1].strip().upper()
                if val.startswith('NEITHER'):
                    return 'Neither'
                if val.startswith('B'):
                    return 'B'
                if val.startswith('A'):
                    return 'A'
        return None

    n_correct   = 0
    error_confs = []
    total = len(PAIRS)

    for stmt_a, stmt_b, wrong_stmt, explanation, is_factual_error in PAIRS:
        response = llm.prompt(PROMPT_TEMPLATE.format(a=stmt_a, b=stmt_b))
        conf = extract_confidence(response)
        verdict = extract_ab_answer(response)
        is_correct = (verdict is not None and verdict.upper() == wrong_stmt.upper())
        if is_correct:
            n_correct += 1
        if is_factual_error and conf is not None:
            error_confs.append(conf)

        kbench.assertions.assert_true(
            is_correct,
            expectation=f"Model must identify '{wrong_stmt}' as the answer. Reason: {explanation}. Got: '{verdict}'"
        )
        kbench.assertions.assert_true(
            conf is not None,
            expectation="Model must output a numeric confidence score 0-100."
        )

    # — Correctness tiers —
    kbench.assertions.assert_true(
        n_correct >= round(0.8 * total),
        expectation=(
            f"[INTERMEDIATE] ≥ 80% correct identification. Got {n_correct}/{total}."
        )
    )

    # — Confidence tiers on clear factual error items —
    if error_confs:
        avg_error_conf = sum(error_confs) / len(error_confs)
        kbench.assertions.assert_true(
            avg_error_conf >= 70,
            expectation=(
                f"[SUCCESS] Avg confidence on clear factual errors = {avg_error_conf:.1f}. "
                "Success requires avg confidence ≥ 70 when the wrong statement is clearly incorrect."
            )
        )
        kbench.assertions.assert_true(
            avg_error_conf >= 50,
            expectation=(
                f"[INTERMEDIATE] Avg confidence on clear factual errors = {avg_error_conf:.1f}. "
                "Intermediate requires avg confidence ≥ 50."
            )
        )

In [ ]:
# Kaggle injects kbench.llm with whichever model was selected in the UI
t06_contradiction_detection.run(llm=kbench.llm)

In [ ]:
# Uncomment to submit best result to the leaderboard
# %choose t06_contradiction_detection